# TP 1 ► Module : Apprentissage Profond (Deep Learning)
> Pr. Gaceb<br>
> S2-Master I2A Pro

**L’Objectif de cette première série de TP** est de Configurer un environnement de développement avec PyCharm, ansi que l'installation des bibliothèques qui serviront au deep learning toute en se familiarisant avec eux*

Les deux bibliothèques les plus utilisées dans ce domaine sont incontestablement **PyTorch (Meta/Facebook)** et **TensorFlow (Google)**.

PyTorch est actuellement la bibliothèque favorite de la recherche et du monde académique, permettant de comprendre en profondeur la structure de vos modèles avec une syntaxe intuitive. Son code ressemble à du Python classique et utilise des graphes de calcul dynamiques, ce qui rend le débogage beaucoup plus simple.

Cette bibliothèque qui sera utilisée majoritairement en TP est idéale pour le prototypage rapide, les projets innovants et l'apprentissage. En revanche, TensorFlow (via Keras) est la référence dominante dans l'industrie et le déploiement en production, permettant de construire des modèles fonctionnels très rapidement avec un minimum de lignes de code.

# Exo 1 - Environment Setup

In [1]:
import torch

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.15.0+cu132
Device: cpu


# Exo 2 $\rightarrow$ 5 [Tensors Tutorial](../../tutorials/Tensors.ipynb)

# Exo 6 - Mise en œuvre de la Descente de Gradient avec PyTorch : Optimisation d'une fonction convexe de degré 2

Pour comprendre la "magie" derrière l'apprentissage automatique et profond. On va utiliser PyTorch pour trouver le minimum de la fonction $f(x)=x^2-4x+4$ avec l’algorithme de la décente de gradient ($x_0=10$, taux d’apprentissage `lr = 0.1`). Mathématiquement, cette fonction est $(x-2)^2$ , donc le minimum est à $2$.

1. **Initialisation :** Déclarez une variable PyTorch `x` initialisée à 10 Assurez-vous que PyTorch puisse calculer les gradients par rapport à cette variable.

In [2]:
x = torch.tensor(10.0, requires_grad=True)
test = x**2  # derivative is 2x
print(f"{x=}")
print(f"{test=}")

x=tensor(10., requires_grad=True)
test=tensor(100., grad_fn=<PowBackward0>)


In [3]:
test.backward()
print(f"{x.grad=}")  # expecting 2 * 10 = 20 for x=10

x.grad=tensor(20.)


2. Appliquer la descente de gradient classique à l’aide d’une boucle à 50 itérations. À chaque étape :
    - Calculez la valeur de la fonction $f(x)=x^2-4x+4$.
    - Déclenchez le calcul des gradients (backward).
    - Mettez à jour x en utilisant la formule mathématique de la descente de gradient :
    $$ x=x – lr × gradient $$
    Afficher l’évolution de `x` chaque 10 itérations.

**Note :** Utilisez with `torch.no_grad()`; pour la mise à jour, et n'oubliez pas de remettre le gradient à zéro.

In [4]:
x = torch.tensor(10.0, requires_grad=True)
lr = 0.1

# 2. Gradient descent loop
for i in range(50):
    f = x**2 - 4*x + 4          # forward pass: compute f(x)
    f.backward()                # backward pass: computes df/dx and stores it in x.grad

    with torch.no_grad():       # the update must not be tracked by autograd (x has `requires_grad=True`)
        x -= lr * x.grad

    x.grad.zero_()              # reset the gradient, otherwise it accumulates

    if (i + 1) % 10 == 0:
        print(f"Iteration {i+1} | x = {x.item():.6f} | f(x) = {f.item():.6f}")

Iteration 10 | x = 2.858993 | f(x) = 1.152921
Iteration 20 | x = 2.092234 | f(x) = 0.013292
Iteration 30 | x = 2.009903 | f(x) = 0.000153
Iteration 40 | x = 2.001063 | f(x) = 0.000002
Iteration 50 | x = 2.000114 | f(x) = 0.000000


3. L'automatisation avec Optimizer : Réinitialisez `x` à 10, utilisez cette fois l'objet [torch.optim.SGD](https://docs.pytorch.org/docs/2.15/generated/torch.optim.SGD.html) pour automatiser la mise à jour à l’aide de la méthode de descente de
gradient stochastique.
- Remplacez votre formule de DG classique par `optimizer.step()`.
- Remplacez la remise à zéro manuelle par `optimizer.zero_grad()`.

Let's define a function for experimenting with different optimizers, learning rates, and number of iterations.

In [5]:
def with_optim(Optimizer, lr=0.1, iterations=50, stream=print):
    stream(f"\n{Optimizer=}\n{iterations=}\n{lr=}\n\n")
    x = torch.tensor(10.0, requires_grad=True)
    optimizer = Optimizer([x], lr=lr)
    
    for i in range(iterations):
        
        f = x**2 - 4*x + 4
        if (i + 1) % 10 == 0:
            stream(f"Iteration {i+1:3d} | x = {x.item(): .6f} | f(x) = {f.item():.6f}")
            
        optimizer.zero_grad()
        f.backward()
        optimizer.step()

    stream(f"\nFinally {x.item()}")

With the `SGD` optimizer:

In [6]:
with_optim(torch.optim.SGD)


Optimizer=<class 'torch.optim.sgd.SGD'>
iterations=50
lr=0.1


Iteration  10 | x =  3.073742 | f(x) = 1.152921
Iteration  20 | x =  2.115292 | f(x) = 0.013292
Iteration  30 | x =  2.012379 | f(x) = 0.000153
Iteration  40 | x =  2.001329 | f(x) = 0.000002
Iteration  50 | x =  2.000143 | f(x) = 0.000000

Finally 2.0001139640808105


4. Application de l’optimiseur Adam : Réinitialisez une nouvelle fois `x` à 10.0.
- Au lieu de SGD, utilisez l'optimiseur [torch.optim.Adam](https://docs.pytorch.org/docs/2.15/generated/torch.optim.Adam.html).
- Gardez le même taux d'apprentissage (`lr=0.1`)
- Observez la vitesse de convergence. Est-elle plus lente ou plus rapide que le SGD sur les premières étapes ?

**Note :** Adam utilise des "moments" (une sorte de mémoire gradients précédents) pour ajuster sa trajectoire. Discuter de la précision de convergence, ajouter des itérations supplémentaires si nécessaire et observer le résultat. Changer la valeur de `lr` et observer le résuat.

In [7]:

from mlkit.util.prettyprint import adjacent

adjacent(
    lambda stream: with_optim(torch.optim.Adam, lr=0.1, iterations=100, stream=stream),
    lambda stream: with_optim(torch.optim.SGD, stream=stream)
)

with_optim(torch.optim.Adam, lr=0.01, iterations=100)
lambda stream: with_optim(torch.optim.Adam, lr=0.5, iterations=100)
lambda stream: with_optim(torch.optim.Adam, lr=0.5, iterations=200)

                                                    |    
Optimizer=<class 'torch.optim.adam.Adam'>           |    Optimizer=<class 'torch.optim.sgd.SGD'>
iterations=100                                      |    iterations=50
lr=0.1                                              |    lr=0.1
                                                    |    
                                                    |    
Iteration  10 | x =  9.103276 | f(x) = 50.456532    |    Iteration  10 | x =  3.073742 | f(x) = 1.152921
Iteration  20 | x =  8.129538 | f(x) = 37.571228    |    Iteration  20 | x =  2.115292 | f(x) = 0.013292
Iteration  30 | x =  7.202230 | f(x) = 27.063196    |    Iteration  30 | x =  2.012379 | f(x) = 0.000153
Iteration  40 | x =  6.340633 | f(x) = 18.841099    |    Iteration  40 | x =  2.001329 | f(x) = 0.000002
Iteration  50 | x =  5.558369 | f(x) = 12.661987    |    Iteration  50 | x =  2.000143 | f(x) = 0.000000
Iteration  60 | x =  4.863608 | f(x) = 8.200251     |    
Iteration  

<function __main__.<lambda>(stream)>